# B0 Image Quality Check

Loads DWI data for one or more HCP 7T subjects, identifies b0 volumes, and checks:
- Number and indices of b0 volumes
- Visual inspection (axial/coronal/sagittal slices)
- Signal consistency across b0 repetitions
- SNR estimate within brain mask
- Signal dropout / outlier volumes

In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from dipy.core.gradients import gradient_table

In [ ]:
DATA_ROOT = "../data"
B0_THRESHOLD = 100

subjects = sorted(os.listdir(DATA_ROOT))
subjects = [s for s in subjects if os.path.isdir(os.path.join(DATA_ROOT, s))]
print(f"Found {len(subjects)} subjects: {subjects}")

## 1. Gradient table — b-value shells and b0 identification

In [ ]:
def load_subject(subject_id, data_root=DATA_ROOT):
    base = os.path.join(data_root, subject_id, "T1w", "Diffusion_7T")
    img   = nib.load(os.path.join(base, "data.nii.gz"))
    data  = img.get_fdata(dtype=np.float32)
    mask  = nib.load(os.path.join(base, "nodif_brain_mask.nii.gz")).get_fdata().astype(bool)
    bvals = np.loadtxt(os.path.join(base, "bvals"))
    bvecs = np.loadtxt(os.path.join(base, "bvecs"))
    gtab  = gradient_table(bvals, bvecs=bvecs, b0_threshold=B0_THRESHOLD)
    return data, mask, gtab, img.affine


# Inspect the first subject
SUBJECT = subjects[0]
data, mask, gtab, affine = load_subject(SUBJECT)

print(f"Subject       : {SUBJECT}")
print(f"Data shape    : {data.shape}")
print(f"Voxel size    : {nib.affines.voxel_sizes(affine).round(3)} mm")
print(f"Total volumes : {data.shape[-1]}")
print(f"b0 threshold  : {B0_THRESHOLD}")
print(f"b0 count      : {gtab.b0s_mask.sum()}")
print(f"b0 indices    : {np.where(gtab.b0s_mask)[0].tolist()}")
print(f"Unique bvals  : {np.unique(np.round(gtab.bvals, -2)).tolist()}")

In [ ]:
# b-value distribution across all volumes
fig, ax = plt.subplots(figsize=(12, 3))
ax.scatter(np.arange(len(gtab.bvals)), gtab.bvals, s=8, c=gtab.b0s_mask, cmap="bwr", zorder=2)
ax.axhline(B0_THRESHOLD, color="k", lw=1, ls="--", label=f"b0 threshold ({B0_THRESHOLD})")
ax.set_xlabel("Volume index")
ax.set_ylabel("b-value (s/mm²)")
ax.set_title(f"b-value per volume — subject {SUBJECT}")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Visual inspection of b0 volumes

In [ ]:
def mid_slices(vol3d):
    """Return axial, coronal, sagittal centre slices."""
    cx, cy, cz = [s // 2 for s in vol3d.shape]
    return vol3d[:, :, cz], vol3d[:, cy, :], vol3d[cx, :, :]


b0_vols = data[:, :, :, gtab.b0s_mask]   # shape (X, Y, Z, N_b0)
n_b0 = b0_vols.shape[-1]

# Show all b0 volumes (axial middle slice)
ncols = min(n_b0, 8)
nrows = int(np.ceil(n_b0 / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5))
axes = np.array(axes).reshape(-1)

vmax = np.percentile(b0_vols, 99)
for i in range(n_b0):
    vol_idx = np.where(gtab.b0s_mask)[0][i]
    axial, _, _ = mid_slices(b0_vols[:, :, :, i])
    axes[i].imshow(axial.T, origin="lower", cmap="gray", vmin=0, vmax=vmax)
    axes[i].set_title(f"vol {vol_idx}", fontsize=9)
    axes[i].axis("off")

for j in range(n_b0, len(axes)):
    axes[j].axis("off")

fig.suptitle(f"All b0 volumes — {SUBJECT} (axial centre slice)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Three-plane view of the mean b0
mean_b0 = b0_vols.mean(axis=-1)
axial, coronal, sagittal = mid_slices(mean_b0)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, slc, title in zip(axes,
                           [axial, coronal, sagittal],
                           ["Axial", "Coronal", "Sagittal"]):
    im = ax.imshow(slc.T, origin="lower", cmap="gray",
                   vmin=0, vmax=np.percentile(mean_b0[mask], 99))
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.04)

fig.suptitle(f"Mean b0 — {SUBJECT}", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Brain mask coverage

In [ ]:
axial_mask, coronal_mask, sagittal_mask = mid_slices(mask.astype(np.float32))
axial_b0,   coronal_b0,   sagittal_b0   = mid_slices(mean_b0)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, b0_slc, mask_slc, title in zip(
        axes,
        [axial_b0, coronal_b0, sagittal_b0],
        [axial_mask, coronal_mask, sagittal_mask],
        ["Axial", "Coronal", "Sagittal"]):
    ax.imshow(b0_slc.T, origin="lower", cmap="gray",
              vmin=0, vmax=np.percentile(mean_b0[mask], 99))
    ax.contour(mask_slc.T, levels=[0.5], colors="red", linewidths=0.8)
    ax.set_title(f"{title} + mask")
    ax.axis("off")

fig.suptitle(f"Brain mask overlay — {SUBJECT}  ({mask.sum():,} voxels)", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Signal consistency across b0 repetitions

In [ ]:
# Per-volume mean signal inside the brain mask
b0_mean_signal = np.array([b0_vols[:, :, :, i][mask].mean() for i in range(n_b0)])
b0_indices     = np.where(gtab.b0s_mask)[0]

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(n_b0), b0_mean_signal, color="steelblue")
ax.axhline(b0_mean_signal.mean(), color="red", ls="--", label=f"mean = {b0_mean_signal.mean():.1f}")
ax.set_xticks(range(n_b0))
ax.set_xticklabels([str(i) for i in b0_indices], fontsize=8)
ax.set_xlabel("Volume index")
ax.set_ylabel("Mean signal (brain)")
ax.set_title(f"Mean b0 signal per volume — {SUBJECT}")
ax.legend()
plt.tight_layout()
plt.show()

# Flag outliers (>2 SD from mean)
deviation = np.abs(b0_mean_signal - b0_mean_signal.mean()) / b0_mean_signal.std()
outliers  = np.where(deviation > 2)[0]
if len(outliers):
    print(f"Potential outlier b0 volumes (>2 SD): {b0_indices[outliers].tolist()}")
else:
    print("No outlier b0 volumes detected (all within 2 SD of mean).")

In [ ]:
# Coefficient of variation (CV) map across b0 repetitions
b0_std  = b0_vols.std(axis=-1)
b0_cv   = np.where(mean_b0 > 1e-6, b0_std / mean_b0, 0.0)

axial_cv, _, _ = mid_slices(b0_cv)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(axial_cv.T, origin="lower", cmap="hot", vmin=0, vmax=0.15)
plt.colorbar(im, ax=ax, label="CV (std / mean)")
ax.set_title(f"b0 coefficient of variation — {SUBJECT}\n(axial centre slice)")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Median CV in brain mask : {b0_cv[mask].median() if hasattr(b0_cv[mask], 'median') else np.median(b0_cv[mask]):.4f}")
print(f"Mean   CV in brain mask : {b0_cv[mask].mean():.4f}")

## 5. SNR estimate

In [ ]:
# SNR = mean_b0 / std_b0 per voxel (temporal SNR across b0 repetitions)
tSNR = np.where(b0_std > 1e-6, mean_b0 / b0_std, 0.0)

axial_tsnr, _, _ = mid_slices(tSNR)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(axial_tsnr.T, origin="lower", cmap="viridis", vmin=0, vmax=80)
plt.colorbar(im, ax=ax, label="tSNR")
ax.set_title(f"Temporal SNR (mean/std across b0s) — {SUBJECT}\n(axial centre slice)")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Median tSNR in brain mask : {np.median(tSNR[mask]):.1f}")
print(f"Mean   tSNR in brain mask : {tSNR[mask].mean():.1f}")

In [ ]:
SLICE_INDEX = [86, 90, 95]   # 单个整数或列表
PLANE       = "axial"        # "axial" / "coronal" / "sagittal"
VOLUME      = "tSNR"         # "tSNR" / "mean_b0" / "cv"

vol_map  = {"tSNR": tSNR, "mean_b0": mean_b0, "cv": b0_cv}
cmap_map = {"tSNR": ("viridis", 0, 80), "mean_b0": ("gray", 0, None), "cv": ("hot", 0, 0.15)}
ax_map   = {"axial": 2, "coronal": 1, "sagittal": 0}
lab_map  = {"axial": ("x", "y"), "coronal": ("x", "z"), "sagittal": ("y", "z")}

vol3d = vol_map[VOLUME]
cmap, vmin, vmax_val = cmap_map[VOLUME]
if vmax_val is None:
    vmax_val = np.percentile(vol3d[mask], 99)
xlabel, ylabel = lab_map[PLANE]

indices = [SLICE_INDEX] if isinstance(SLICE_INDEX, int) else SLICE_INDEX

fig, axes = plt.subplots(1, len(indices), figsize=(5 * len(indices), 5))
axes = np.array(axes).reshape(-1)

for ax, idx in zip(axes, indices):
    if PLANE == "axial":
        slc = vol3d[:, :, idx]
    elif PLANE == "coronal":
        slc = vol3d[:, idx, :]
    else:
        slc = vol3d[idx, :, :]
    im = ax.imshow(slc.T, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax_val)
    ax.set_title(f"{PLANE} {idx}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.colorbar(im, ax=ax, fraction=0.04)

fig.suptitle(f"{VOLUME} — {SUBJECT}", fontsize=13)
plt.tight_layout()
plt.show()

print(f"{PLANE} 方向共 {vol3d.shape[ax_map[PLANE]]} 层")

In [ ]:
# Signal histogram for mean b0 (inside brain mask)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(mean_b0[mask], bins=100, color="steelblue", edgecolor="none", alpha=0.85)
ax.set_xlabel("Signal intensity")
ax.set_ylabel("Voxel count")
ax.set_title(f"Mean b0 signal distribution (brain mask) — {SUBJECT}")
plt.tight_layout()
plt.show()

## 6. Cross-subject summary

In [ ]:
rows = []
for subj in subjects:
    try:
        d, m, g, _ = load_subject(subj)
        b0s = d[:, :, :, g.b0s_mask]
        mn  = b0s.mean(axis=-1)
        sd  = b0s.std(axis=-1)
        snr = np.where(sd > 1e-6, mn / sd, 0.0)
        rows.append({
            "subject":        subj,
            "shape":          str(d.shape),
            "n_volumes":      d.shape[-1],
            "n_b0":           int(g.b0s_mask.sum()),
            "brain_voxels":   int(m.sum()),
            "mean_b0_signal": float(mn[m].mean()),
            "median_tSNR":    float(np.median(snr[m])),
        })
        print(f"  {subj} OK")
    except Exception as e:
        print(f"  {subj} FAILED: {e}")
        rows.append({"subject": subj, "error": str(e)})

df_summary = pd.DataFrame(rows)
df_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ok = df_summary.dropna(subset=["median_tSNR"])

axes[0].bar(ok["subject"], ok["mean_b0_signal"], color="steelblue")
axes[0].set_xticklabels(ok["subject"], rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Mean signal")
axes[0].set_title("Mean b0 signal per subject")

axes[1].bar(ok["subject"], ok["median_tSNR"], color="darkorange")
axes[1].set_xticklabels(ok["subject"], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Median tSNR")
axes[1].set_title("Temporal SNR per subject")

plt.tight_layout()
plt.show()